In [5]:
!pip -q install pymupdf requests pandas

In [6]:
import os
import json
import time
import re
import requests
import pymupdf
import pandas as pd
import xml.etree.ElementTree as ET

from google.colab import drive

drive.mount("/content/drive")

folder = "/content/drive/MyDrive/IR_Project_Final"
pdf_folder = folder + "/pdfs"
text_folder = folder + "/texts"

os.makedirs(pdf_folder, exist_ok=True)
os.makedirs(text_folder, exist_ok=True)

print("Setup completed!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup completed!


In [7]:
subtopics = {
    "Language Identification": "language identification",
    "Text Classification": "text classification",
    "Sentiment Analysis": "sentiment analysis",
    "Text Summarization": "text summarization"
}

papers_per_topic = 100

print("Subtopics are ready!")

Subtopics are ready!


In [4]:
api_url = "https://export.arxiv.org/api/query"
metadata_file = folder + "/papers_metadata.json"

# Load saved papers if they exist
if os.path.exists(metadata_file):
    with open(metadata_file, "r", encoding="utf-8") as f:
        papers = json.load(f)
else:
    papers = []

seen_ids = {p["arxiv_id"] for p in papers}

for topic, keyword in subtopics.items():

    count = sum(p["subtopic"] == topic for p in papers)
    start = 0

    print("Collecting:", topic)

    while count < papers_per_topic:

        params = {
            "search_query": f'cat:cs.CL AND all:"{keyword}"',
            "start": start,
            "max_results": 100,
            "sortBy": "submittedDate",
            "sortOrder": "descending"
        }

        try:
            response = requests.get(api_url, params=params, timeout=90)
            response.raise_for_status()

            root = ET.fromstring(response.content)
            entries = root.findall("{http://www.w3.org/2005/Atom}entry")

            if not entries:
                print("No more results for:", topic)
                break

            for entry in entries:
                link = entry.find("{http://www.w3.org/2005/Atom}id").text
                paper_id = re.sub(r"v\d+$", "", link.split("/abs/")[-1])

                if paper_id in seen_ids:
                    continue

                title = entry.find("{http://www.w3.org/2005/Atom}title").text
                abstract = entry.find("{http://www.w3.org/2005/Atom}summary").text

                papers.append({
                    "arxiv_id": paper_id,
                    "title": " ".join(title.split()),
                    "abstract": " ".join(abstract.split()),
                    "arxiv_url": link,
                    "pdf_url": f"https://arxiv.org/pdf/{paper_id}",
                    "subtopic": topic
                })

                seen_ids.add(paper_id)
                count += 1

                if count == papers_per_topic:
                    break

            with open(metadata_file, "w", encoding="utf-8") as f:
                json.dump(papers, f, indent=2, ensure_ascii=False)

            start += 100

            if count < papers_per_topic:
                time.sleep(5)

        except Exception as e:
            print("Error:", e)
            break

    print(topic, ":", count)
    time.sleep(5)

print("\nTotal unique papers:", len(seen_ids))

Collecting: Language Identification
Language Identification : 100
Collecting: Text Classification
Text Classification : 100
Collecting: Sentiment Analysis
Sentiment Analysis : 100
Collecting: Text Summarization
Text Summarization : 100

Total unique papers: 400


In [8]:
processed_file = folder + "/processed_papers.json"

# Load saved progress
if os.path.exists(processed_file):
    with open(processed_file, "r", encoding="utf-8") as f:
        processed = json.load(f)
else:
    processed = []

completed_ids = {p["arxiv_id"] for p in processed}

# Make sure we collected 400 unique papers
assert len({p["arxiv_id"] for p in papers}) >= 400, "Collect 400 unique papers first!"

for paper in papers:

    paper_id = paper["arxiv_id"]

    if paper_id in completed_ids:
        continue

    pdf_path = pdf_folder + "/" + paper_id + ".pdf"
    text_path = text_folder + "/" + paper_id + ".txt"

    try:
        # Download PDF
        if not os.path.exists(pdf_path):
            response = requests.get(paper["pdf_url"], timeout=90)
            response.raise_for_status()

            with open(pdf_path, "wb") as f:
                f.write(response.content)

        # Extract text
        with pymupdf.open(pdf_path) as doc:
            page_count = len(doc)
            text = "\n".join(page.get_text() for page in doc)

        if len(text.strip()) < 100:
            print("Skipped:", paper_id)
            continue

        # Save text
        with open(text_path, "w", encoding="utf-8") as f:
            f.write(text)

        # Save paper information
        processed.append({
            **paper,
            "page_count": page_count
        })

        completed_ids.add(paper_id)

        # Save progress
        with open(processed_file, "w", encoding="utf-8") as f:
            json.dump(processed, f, indent=2, ensure_ascii=False)

        print(f"{len(processed)}/400 - {paper_id}")

        time.sleep(2)

    except Exception as e:
        print("Failed:", paper_id, "-", e)

print("\nSuccessfully processed:", len(processed))


Successfully processed: 400


In [9]:
# Classify papers based on page count
for paper in processed:

    pages = paper["page_count"]

    if pages < 6:
        paper["length_type"] = "Short"
    elif 7 <= pages <= 9:
        paper["length_type"] = "Medium"
    elif pages > 9:
        paper["length_type"] = "Long"
    else:
        paper["length_type"] = "Unclassified"

# Save updated papers
with open(processed_file, "w", encoding="utf-8") as f:
    json.dump(processed, f, indent=2, ensure_ascii=False)

# Create DataFrame
df = pd.DataFrame(processed)

# Show statistics
print("Total papers:", len(df))

print("\nPapers per subtopic:")
print(df["subtopic"].value_counts())

print("\nPapers by document length:")
print(df["length_type"].value_counts())

display(df.head())

Total papers: 400

Papers per subtopic:
subtopic
Language Identification    100
Text Classification        100
Sentiment Analysis         100
Text Summarization         100
Name: count, dtype: int64

Papers by document length:
length_type
Long            292
Medium           55
Short            27
Unclassified     26
Name: count, dtype: int64


,arxiv_id,title,abstract,arxiv_url,pdf_url,subtopic,page_count,length_type
0,2610.32382,Acoustic and Lexical Signals for Native Langua...,Spoken native language (L1) identification for...,http://arxiv.org/abs/2610.32382v1,https://arxiv.org/pdf/2610.32382,Language Identification,8,Medium
1,2610.25505,Where Are Language-Specific Features Represent...,Previous work has argued that multilingual LLM...,http://arxiv.org/abs/2610.25505v1,https://arxiv.org/pdf/2610.25505,Language Identification,15,Long
2,2610.23309,Inferring L1 Backgrounds from L2 English with ...,Native language identification (NLI) infers a ...,http://arxiv.org/abs/2610.23309v1,https://arxiv.org/pdf/2610.23309,Language Identification,8,Medium
3,2610.21941,Learning Speech Attribute Representations via ...,Recent post-training approaches have shown tha...,http://arxiv.org/abs/2610.21941v1,https://arxiv.org/pdf/2610.21941,Language Identification,8,Medium
4,2610.20263,Anuvani-LID: Robust and Calibrated Spoken Lang...,Every multilingual speech pipeline begins with...,http://arxiv.org/abs/2610.20263v2,https://arxiv.org/pdf/2610.20263,Language Identification,25,Long


In [10]:
from datetime import datetime, timezone

collection_info = {
    "source": "arXiv API",
    "category": "cs.CL",
    "subtopics": subtopics,
    "total_papers": len(processed),
    "queries": [
        f'cat:cs.CL AND all:"{keyword}"'
        for keyword in subtopics.values()
    ],
    "sort_by": "submittedDate",
    "sort_order": "descending",
    "date_filter": "No date filter",
    "recorded_at": datetime.now(timezone.utc).isoformat()
}

with open(folder + "/collection_info.json", "w", encoding="utf-8") as f:
    json.dump(collection_info, f, indent=2, ensure_ascii=False)

print("Collection information saved!")

Collection information saved!


In [11]:
!pip -q install rank-bm25

In [12]:
import re
from difflib import get_close_matches
from collections import Counter
from rank_bm25 import BM25Okapi

# Load papers and their text
documents = []
titles = []
links = []

for paper in processed:
    path = text_folder + "/" + paper["arxiv_id"] + ".txt"

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            documents.append(f.read())

        titles.append(paper["title"])
        links.append(paper["arxiv_url"])

# Tokenization and case insensitivity
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_documents = [tokenize(text) for text in documents]

# Build BM25
bm25 = BM25Okapi(tokenized_documents)

# Vocabulary for spelling correction
word_counts = Counter(
    word for doc in tokenized_documents for word in doc
)

vocabulary = {
    word for word, count in word_counts.items()
    if count >= 2
}

# Abbreviation expansion
abbreviations = {
    "nlp": "natural language processing",
    "ai": "artificial intelligence",
    "ml": "machine learning",
    "llm": "large language model"
}

# Process user query
def prepare_query(query):
    words = tokenize(query)

    corrected = []

    for word in words:
        if word in abbreviations:
            corrected.extend(tokenize(abbreviations[word]))

        elif word in vocabulary:
            corrected.append(word)

        else:
            match = get_close_matches(word, vocabulary, n=1, cutoff=0.85)
            corrected.append(match[0] if match else word)

    return " ".join(corrected)

print("BM25 is ready!")
print("Indexed documents:", len(documents))

BM25 is ready!
Indexed documents: 400


In [13]:
def search_bm25(query):

    # Extract phrases inside quotation marks
    phrases = re.findall(r'"([^"]+)"', query)

    # Process query
    clean_query = prepare_query(query)

    # Calculate BM25 scores
    scores = bm25.get_scores(tokenize(clean_query))

    # Rank documents by score
    ranked = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    # Exact phrase matching
    if phrases:
        corrected_phrases = [
            prepare_query(phrase).split()
            for phrase in phrases
        ]

        ranked = [
            i for i in ranked
            if all(
                any(
                    tokenized_documents[i][j:j + len(phrase)] == phrase
                    for j in range(len(tokenized_documents[i]) - len(phrase) + 1)
                )
                for phrase in corrected_phrases
            )
        ]

    # Display top 10 results
    print("Original query:", query)
    print("Processed query:", clean_query)
    print()

    for i in ranked[:10]:
        print("Title:", titles[i])
        print("Link:", links[i])
        print("Snippet:", " ".join(documents[i].split())[:200], "...")
        print("BM25 Score:", round(float(scores[i]), 3))
        print("-" * 50)


# Test the search engine
search_bm25('"sentimant analysis"')

Original query: "sentimant analysis"
Processed query: sentiment analysis

Title: FinMoji: A Framework for Emoji-driven Sentiment Analysis in Financial Social Media
Link: http://arxiv.org/abs/2605.09469v1
Snippet: FinMoji: A Framework for Emoji-driven Sentiment Analysis in Financial Social Media AHMED MAHROUS, King Abdullah University of Science and Technology (KAUST), Saudi Arabia JENS SCHNEIDER, Hamad Bin Kha ...
BM25 Score: 3.441
--------------------------------------------------
Title: Unveiling Public Opinion: A Study of Sentiment Analysis Using LSTM and Traditional Models
Link: http://arxiv.org/abs/2607.07772v1
Snippet: © 2025 IEEE. Personal use of this material is permitted. Permission from IEEE must be obtained for all other uses, in any current or future media, including reprinting/republishing this material for a ...
BM25 Score: 3.441
--------------------------------------------------
Title: Evaluating LLM Usage for Efficient and Explainable Numerical and Classified Implicit S

In [14]:
!pip -q install sentence-transformers

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pretrained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Use the same documents as BM25
dense_documents = [
    titles[i] + ". " + documents[i][:4000]
    for i in range(len(documents))
]

# Create embeddings
embeddings_file = folder + "/document_embeddings.npy"

if os.path.exists(embeddings_file):
    document_embeddings = np.load(embeddings_file)

    if len(document_embeddings) != len(documents):
        document_embeddings = model.encode(
            dense_documents,
            batch_size=16,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        np.save(embeddings_file, document_embeddings)

else:
    document_embeddings = model.encode(
        dense_documents,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(embeddings_file, document_embeddings)

print("Dense Retrieval is ready!")
print("Document embeddings:", document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense Retrieval is ready!
Document embeddings: (400, 384)


In [21]:
def search_dense(query):

    clean_query = prepare_query(query)

    # Convert query into an embedding
    query_embedding = model.encode(
        clean_query,
        normalize_embeddings=True
    )

    # Calculate similarity
    scores = document_embeddings @ query_embedding

    # Get top 10 papers
    top_indices = np.argsort(scores)[::-1][:10]

    for i in top_indices:
        print("Title:", titles[i])
        print("Link:", links[i])
        print("Score:", round(float(scores[i]), 3))
        print("-" * 50)


# Test
search_dense("sentiment analysis")

Title: From TF-IDF to Transformers: A Comparative and Ensemble Approach to Sentiment Classification
Link: http://arxiv.org/abs/2605.22003v1
Score: 0.607
--------------------------------------------------
Title: SalAngaBhava: A Sinhala Market Dataset for Aspect-based Sentiment Analysis
Link: http://arxiv.org/abs/2607.05259v1
Score: 0.571
--------------------------------------------------
Title: Unveiling Public Opinion: A Study of Sentiment Analysis Using LSTM and Traditional Models
Link: http://arxiv.org/abs/2607.07772v1
Score: 0.565
--------------------------------------------------
Title: Language-Specific Sentiment Polarity Biases in Encoder and Large Language Model Classification of Product Reviews
Link: http://arxiv.org/abs/2606.22745v1
Score: 0.55
--------------------------------------------------
Title: Hybrid TF--IDF Logistic Regression and MLP Neural Baseline for Indonesian Three-Class Sentiment Analysis on Social Media Text
Link: http://arxiv.org/abs/2605.07793v1
Score: 0.543

In [17]:
def search_hybrid(query):

    # Process query
    clean_query = prepare_query(query)

    # BM25 scores
    bm25_scores = bm25.get_scores(tokenize(clean_query))

    # Dense scores
    query_embedding = model.encode(
        clean_query,
        normalize_embeddings=True
    )
    dense_scores = document_embeddings @ query_embedding

    # Rank documents
    bm25_rank = np.argsort(bm25_scores)[::-1]
    dense_rank = np.argsort(dense_scores)[::-1]

    # Reciprocal Rank Fusion
    rrf_scores = np.zeros(len(documents))

    for rank, i in enumerate(bm25_rank):
        rrf_scores[i] += 1 / (60 + rank + 1)

    for rank, i in enumerate(dense_rank):
        rrf_scores[i] += 1 / (60 + rank + 1)

    # Exact phrase matching
    phrases = re.findall(r'"([^"]+)"', query)

    final_rank = np.argsort(rrf_scores)[::-1]

    if phrases:
        corrected_phrases = [
            prepare_query(phrase).split()
            for phrase in phrases
        ]

        final_rank = [
            i for i in final_rank
            if all(
                any(
                    tokenized_documents[i][j:j + len(phrase)] == phrase
                    for j in range(len(tokenized_documents[i]) - len(phrase) + 1)
                )
                for phrase in corrected_phrases
            )
        ]

    # Display top 10 results
    print("Original query:", query)
    print("Processed query:", clean_query)
    print()

    for i in final_rank[:10]:
        print("Title:", titles[i])
        print("Link:", links[i])
        print("Snippet:", " ".join(documents[i].split())[:200], "...")
        print("RRF Score:", round(float(rrf_scores[i]), 4))
        print("-" * 50)


# Test
search_hybrid("sentiment analysis")

Original query: sentiment analysis
Processed query: sentiment analysis

Title: Unveiling Public Opinion: A Study of Sentiment Analysis Using LSTM and Traditional Models
Link: http://arxiv.org/abs/2607.07772v1
Snippet: © 2025 IEEE. Personal use of this material is permitted. Permission from IEEE must be obtained for all other uses, in any current or future media, including reprinting/republishing this material for a ...
RRF Score: 0.032
--------------------------------------------------
Title: SalAngaBhava: A Sinhala Market Dataset for Aspect-based Sentiment Analysis
Link: http://arxiv.org/abs/2607.05259v1
Snippet: SalAngaBhava: A Sinhala Market Dataset for Aspect-based Sentiment Analysis Lakshani Galwatta∗, Nisansa de Silva∗, Sarangi Aththanayake∗, Adithya Galwatta† ∗Dept. of Computer Science & Engineering, Uni ...
RRF Score: 0.0313
--------------------------------------------------
Title: Beyond Sentiment: Comparing Traditional NLP and LLM-Based Multi-Dimensional Analysis for Politica

In [18]:
!pip -q install gradio

In [22]:
import gradio as gr
import html

# Highlight search terms in yellow
def highlight(text, query):
    words = tokenize(query)
    pattern = r"\b(" + "|".join(
        re.escape(w) for w in sorted(set(words), key=len, reverse=True)
    ) + r")\b"

    safe_text = html.escape(text)
    if not words:
        return safe_text

    return re.sub(
        pattern,
        lambda m: f'<mark style="background-color:yellow;color:black;">{m.group(0)}</mark>',
        safe_text,
        flags=re.IGNORECASE
    )


def search_gui(query, method):

    if not query.strip():
        return "Please enter a search query."

    # Process query
    clean_query = prepare_query(query)

    # BM25 scores
    bm25_scores = bm25.get_scores(tokenize(clean_query))

    # Dense scores
    query_embedding = model.encode(
        clean_query,
        normalize_embeddings=True
    )
    dense_scores = document_embeddings @ query_embedding

    # Select retrieval method
    if method == "BM25":
        scores = bm25_scores

    elif method == "Dense":
        scores = dense_scores

    else:
        bm25_rank = np.argsort(bm25_scores)[::-1]
        dense_rank = np.argsort(dense_scores)[::-1]

        scores = np.zeros(len(documents))

        for rank, i in enumerate(bm25_rank):
            scores[i] += 1 / (60 + rank + 1)

        for rank, i in enumerate(dense_rank):
            scores[i] += 1 / (60 + rank + 1)

    # Rank results
    ranked = np.argsort(scores)[::-1]

    # Exact phrase matching
    phrases = re.findall(r'"([^"]+)"', query)

    if phrases:
        corrected_phrases = [
            prepare_query(phrase).split()
            for phrase in phrases
        ]

        ranked = [
            i for i in ranked
            if all(
                any(
                    tokenized_documents[i][j:j + len(phrase)] == phrase
                    for j in range(len(tokenized_documents[i]) - len(phrase) + 1)
                )
                for phrase in corrected_phrases
            )
        ]

    # Display top 10 results
    results = f"<h3>Search Results ({html.escape(method)})</h3>"
    results += f"<p><b>Processed query:</b> {html.escape(clean_query)}</p>"

    for i in ranked[:10]:
        snippet = " ".join(documents[i].split())[:200]

        # Highlight matching words
        title = highlight(titles[i], clean_query)
        snippet = highlight(snippet, clean_query)

        results += f"<p><b>{title}</b></p>"
        results += f'<p><a href="{html.escape(links[i], quote=True)}" target="_blank">View on arXiv</a></p>'
        results += f"<p><b>Snippet:</b> {snippet}...</p>"
        results += f"<p><b>Score:</b> {float(scores[i]):.4f}</p>"
        results += "<hr>"

    if len(ranked) == 0:
        results += "No matching papers found."

    return results


demo = gr.Interface(
    fn=search_gui,
    inputs=[
        gr.Textbox(label="Search Query", placeholder='Try: "sentiment analysis"'),
        gr.Dropdown(
            choices=["Hybrid (RRF)", "BM25", "Dense"],
            value="Hybrid (RRF)",
            label="Retrieval Method"
        )
    ],
    outputs=gr.HTML(label="Top 10 Results"),
    title="arXiv Hybrid Retrieval Engine",
    description="Search 400 NLP research papers using BM25, Dense Retrieval, or Hybrid RRF."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0948281d7f5d166ac5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
